# Lecture 9: Standalone Wigner Cat Phases: Inverse Participation Ratios


    by M.Süzen
    (c) 2026

This lecture demonstrates Wigner Cat phases in a vanilla case without using mixed ensembles or any package.  See references [suzen26, suzen26zen]

1. `State permutation scheme`: We showing that bimodal distributions for the density is not a numerical artifact and as well as not depend on the choose of the enumeration of  
2. Inverse participation distributions: This supports the evidence of localisation. 


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

font = {"family": "Tahoma", "weight": "bold", "size": 19}

plt.rc("font", **font)

## State permutation scheme

As mentioned in the paper, the block diagonal structure can be permuted and we cat still 
get bimodal eigendensity given selection parameter (order parameter). We demonstrated this 
via scrambling of states, columns on the different blocks. We scramble such that block structure remains intact, so only blocks are shuffled.

**Exercise 1**: Use portions of the function `generate_block_preserved_wigner_cat`, and check if using the same permutation matrix won't change the results.  

In [ ]:
def generate_block_preserved_wigner_cat(N=1000, mu=0.7, test_mode="original"):
    """
    Replicates and validates the Wigner Cat Phase while strictly preserving
    the block-diagonal structure with perfect zero blocks.

    test_mode options:
      'original'    : Standard block diagonal matrix layout.
      'intra_scramble': Randomizes state enumeration independently WITHIN each block,
                      keeping the off-diagonal blocks completely zero.
    """
    L = int(mu * N)

    # Generate the single chaotic matrix block H0 (GOE scale)
    A = np.random.normal(0, 1, (L, L))
    H0 = (A + A.T) / 2.0

    if test_mode == "original":
        # Your exact physical block layout
        H_composite = np.block([[H0, np.zeros((L, L))], [np.zeros((L, L)), H0]])

    elif test_mode == "intra_scramble":
        # Generate two separate permutation matrices for inside the blocks
        p1 = np.random.permutation(L)
        p2 = np.random.permutation(L)
        P1 = np.eye(L)[p1]
        P2 = np.eye(L)[p2]

        # Scramble the inner basis states of each block independently
        H0_block1 = P1 @ H0 @ P1.T
        H0_block2 = P2 @ H0 @ P2.T

        # Assemble the composite matrix keeping off-diagonals perfectly zero
        H_composite = np.block(
            [[H0_block1, np.zeros((L, L))], [np.zeros((L, L)), H0_block2]]
        )

    # Compute the eigenvalues without sorting
    unsorted_spectrum = np.real(np.linalg.eigvals(H_composite))

    # Truncate at exactly N
    return unsorted_spectrum[:N]


# --- Run the Block-Preserved Test ---
np.random.seed(42)
N_target = 1500
mu_target = 0.7

spectrum_orig = generate_block_preserved_wigner_cat(
    N=N_target, mu=mu_target, test_mode="original"
)
spectrum_block_safe = generate_block_preserved_wigner_cat(
    N=N_target, mu=mu_target, test_mode="intra_scramble"
)

# --- Render and Compare Profiles ---
plt.figure(figsize=(5, 12))

# Plot 1: Original Code
plt.subplot(2, 1, 1)
plt.hist(
    spectrum_orig, bins=60, density=True, color="crimson", edgecolor="black", alpha=0.8
)
plt.title(
    "1. Original Block Matrix\n(Standard Execution)",
    fontsize=18,
)
plt.xlabel("Energy Level", fontsize=19)
plt.ylabel("Density", fontsize=19)
plt.grid(True, linestyle=":", alpha=0.4)

# Plot 2: Block-Preserved Randomization
plt.subplot(2, 1, 2)
plt.hist(
    spectrum_block_safe,
    bins=60,
    density=True,
    color="darkorange",
    edgecolor="black",
    alpha=0.8,
)
plt.title("2. Intra-Block Scrambled Matrix\n(Zeros Strictly Maintained)", fontsize=18)
plt.xlabel("Energy Level", fontsize=19)
plt.grid(True, linestyle=":", alpha=0.4)

plt.tight_layout()

In [ ]:
plt.close()

## Inverse Participation Ratios (IPR) distributions

To validate state localization or bimodality, we use IPR distributions.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

fs = 23
# =====================================================================
# GLOBAL FONTS CONFIGURATION FOR PUBLICATION (SIZE 19)
# =====================================================================
font_config = {"family": "Tahoma", "weight": "normal", "size": fs}
plt.rc("font", **font_config)


def generate_asymmetric_wigner_cat_ipr_pipeline(n=1000, mu=0.7):
    """
    Wigner Cat IPR pipeline with spatial eigenvector truncation.
    Truncates eigenvector components to N rows, breaking the blocks into
    sizes n_i and (N - n_i) respectively, producing a bimodal IPR distribution.
    """
    # 1. Determine sub-block size from the mu ratio
    n_i = int(mu * n)
    total_dim = 2 * n_i

    # 2. Initialize matrix block at the sub-block size (n_i by n_i)
    a_block = np.random.normal(0, 1, (n_i, n_i))
    h0 = (a_block + a_block.T) / 2.0  # Symmetric sub-block matrix

    # 3. Create independent permutation matrices per block matching size n_i
    p1 = np.random.permutation(n_i)
    p2 = np.random.permutation(n_i)
    P1 = np.eye(n_i)[p1]
    P2 = np.eye(n_i)[p2]

    # 4. Scramble inner basis states of each block independently
    h0_block1 = P1 @ h0 @ P1.T
    h0_block2 = P2 @ h0 @ P2.T

    # 5. Assemble full composite matrix entity (2*n_i by 2*n_i)
    h_composite_full = np.zeros((total_dim, total_dim))
    h_composite_full[:n_i, :n_i] = h0_block1  # Block 1
    h_composite_full[n_i:, n_i:] = h0_block2  # Block 2

    # 6. Use 'eig' to maintain block structure order
    evals, evecs = np.linalg.eig(h_composite_full)

    # 7. Strip complex types immediately
    evals = np.real(evals)
    evecs = np.real(evecs)

    # 8. Truncate both columns (states) AND rows (basis components) to N
    truncated_evals = evals[:n]
    truncated_evecs = evecs[:n, :n]  # Shape becomes (N, N)

    # 9. Re-normalize eigenvectors since row truncation alters their lengths
    norms = np.linalg.norm(truncated_evecs, axis=0)
    norms[norms == 0] = 1.0  # Safeguard against division by zero
    normalized_evecs = truncated_evecs / norms

    # 10. Compute microstate Inverse Participation Ratio (IPR)
    ipr_values = np.sum(normalized_evecs**4, axis=0)

    # Separate the IPR values by their block origin for explicit visualization
    block1_ipr = ipr_values[:n_i]
    block2_ipr = ipr_values[n_i:]

    return truncated_evals, ipr_values, block1_ipr, block2_ipr, n_i


# =====================================================================
# CELL EXECUTION, MARKS & EXPORT PARAMETERS
# =====================================================================

# Configuration Parameters
np.random.seed(42)
n_size = 1000
mu_ratio = 0.65

# Run simulation pipeline
evals, total_ipr, b1_ipr, b2_ipr, n_1 = generate_asymmetric_wigner_cat_ipr_pipeline(
    n=n_size, mu=mu_ratio
)
n_2_truncated = n_size - n_1

# Analytical GOE Prediction Targets
theoretical_ipr_b1 = 3.0 / n_1
theoretical_ipr_b2 = 3.0 / n_2_truncated

# Initialize the notebook subplots in a vertical stack (Expanded size for high-res layouts)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 15))

# -----------------------------------------------------------------
# Row 1: Density of States (DOS) Histogram
# -----------------------------------------------------------------
ax1.hist(evals, bins=80, density=True, color="crimson", edgecolor="black", alpha=0.8)
ax1.set_title(
    "Eigendensity\n",
    fontname="Tahoma",
    fontweight="bold",
    fontsize=fs,
    pad=15,
)
ax1.set_xlabel("Energy Level", fontname="Tahoma", fontsize=fs)
ax1.set_ylabel("Spectral Density", fontname="Tahoma", fontsize=fs)
ax1.grid(True, linestyle=":", alpha=0.4)
ax1.tick_params(axis="both", which="major", labelsize=fs)

# -----------------------------------------------------------------
# Row 2: Bimodal IPR Distribution P(I_4) vs I_4
# -----------------------------------------------------------------
ax2.hist(
    b1_ipr,
    bins=40,
    density=True,
    color="dodgerblue",
    alpha=0.6,
    edgecolor="blue",
    label=f"Block 1 (Extended over large subspace, $n_1$={n_1})",
)
ax2.hist(
    b2_ipr,
    bins=40,
    density=True,
    color="gold",
    alpha=0.6,
    edgecolor="orange",
    label=f"Block 2 (State Freezing / Localized in truncated window, $N-n_1$={n_2_truncated})",
)
ax2.hist(
    total_ipr,
    bins=80,
    density=True,
    histtype="step",
    color="black",
    linewidth=2.0,
    label="Total System $P(I_4)$ Bimodal MBL Profile",
)

# Add Theoretical GOE Markers
ax2.axvline(
    theoretical_ipr_b1,
    color="darkblue",
    linestyle="--",
    linewidth=2.5,
    label=f"GOE Target 1 (Extended): 3/$n_1$ ({theoretical_ipr_b1:.5f})",
)
ax2.axvline(
    theoretical_ipr_b2,
    color="darkred",
    linestyle="--",
    linewidth=2.5,
    label=f"GOE Target 2 (Localized): 3/($N-n_1$) ({theoretical_ipr_b2:.5f})",
)
ax2.set_title(
    "Bimodal IPR Raw Distribution\n",
    fontname="Tahoma",
    fontweight="bold",
    fontsize=fs,
    pad=15,
)
ax2.set_xlabel("Inverse Participation Ratio ($I_4$)", fontname="Tahoma", fontsize=fs)
ax2.set_ylabel("Probability Density $P(I_4)$", fontname="Tahoma", fontsize=fs)
ax2.grid(True, linestyle=":", alpha=0.4)
ax2.tick_params(axis="both", which="major", labelsize=fs)

# Configure Legend with a legible sizing threshold for papers
ax2.legend(frameon=True, fontsize=17, loc="upper right")

plt.tight_layout()


**Exercise 2** Change the selection ratio, order parameter, when do you see two "cat-ears" start to come togather?

## Reference 

[suzen26] Wigner Cat Phases: A finely tunable system for exploring the transition to quantum chaos [arXiv:2512.22169](https://arxiv.org/abs/2512.22169)    

[suzen26zen] Zenodo Supplement: Wigner Cat Phases: A finely tunable system for exploring the transition to quantum chaos [here](https://zenodo.org/records/20638518).
